# Smart MCQ Solver Challenge, final notebook

Course: Deep Learning and Generative AI Project, Diploma Level  
Roll number: 22f3000114  
Competition: Smart MCQ Solver Challenge  
Metric: MAP@3

This notebook contains the full pipeline end to end: data checks, cleaning, and three models of
different architectures. It runs in two modes.

**Build mode.** Set `BUILD = True`. Trains the from scratch model, runs retrieval, computes
probabilities for every Llama signal, and saves all of it to `/kaggle/working`. Run this once.

**Submit mode.** Set `BUILD = False`. Loads the saved arrays and writes `submission.csv` for
whichever model is selected in `SUBMIT_MODEL`. Runs in under a minute and needs no GPU.
Three submissions are made by changing `SUBMIT_MODEL` and rerunning.

## The three models

| Model | Type | Section |
|---|---|---|
| BiLSTM classifier | Built from scratch, no pretrained weights | 5 |
| Llama 3.1 8B with retrieval | Pretrained, 4 bit quantised, RAG context | 6 |
| Four signal ensemble | Base and QLoRA tuned, with and without context | 7 |

## Inputs to attach

1. Competition data
2. `jjinho/wikipedia-2023-07-faiss-index`
3. `jjinho/wikipedia-20230701`

Accelerator: GPU T4 x2. The `WANDB` secret is needed for experiment tracking.

## 1. Setup

Packages are installed only when missing. The Kaggle image already ships most of them, and a
plain `pip install` on every run wastes minutes of the session.

In [1]:
!pip install -q bitsandbytes peft faiss-cpu langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.0 MB/s eta 0:00:00:00:010:01


In [2]:
import os, gc, re, glob, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

OPTIONS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP = {o: i for i, o in enumerate(OPTIONS)}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| GPUs:', torch.cuda.device_count())

Device: cuda | GPUs: 2


Two switches control the whole notebook. `BUILD` decides whether the expensive cells run or the
saved artifacts are loaded. `SUBMIT_MODEL` picks which model writes `submission.csv`.

In [3]:
BUILD = True
SUBMIT_MODEL = 'base_rag'   # bilstm | base_closed | base_rag | tuned_closed | tuned_rag | ensemble

WORK = '/kaggle/working'
ART  = WORK if BUILD else '/kaggle/input/dl-22f3000114-artifacts'

TRAIN_CSV  = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_CSV   = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
WIKI_INDEX = '/kaggle/input/datasets/jjinho/wikipedia-2023-07-faiss-index/wikipedia_202307.index'
WIKI_MAP   = '/kaggle/input/datasets/jjinho/wikipedia-20230701/wiki_2023_index.parquet'
WIKI_DIR   = '/kaggle/input/datasets/jjinho/wikipedia-20230701'

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
USE_ADAPTER    = True
ADAPTER_HANDLE = 'dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter'
NEED_RETRIEVAL = False          # set for real in section 7

TOKENIZER_NAME  = 'microsoft/deberta-v3-small'
BI_ENCODER_NAME = 'all-MiniLM-L6-v2'
RERANKER_NAME   = 'cross-encoder/ms-marco-MiniLM-L6-v2'
LLAMA_NAME      = 'meta-llama/Llama-3.1-8B-Instruct'

K_ARTICLES, K_RERANK, K_FINAL, CTX_CHARS = 10, 20, 5, 4500
CTX_CACHE = f'{WORK}/contexts_final.pkl'

MAX_LEN, EMBED_DIM, HIDDEN_DIM = 512, 128, 256
N_EPOCHS, BATCH_SIZE, LR       = 20, 32, 1e-3

print('BUILD =', BUILD, '| SUBMIT_MODEL =', SUBMIT_MODEL, '| ART =', ART)

BUILD = True | SUBMIT_MODEL = base_rag | ART = /kaggle/working


## 2. Load the competition data

In [4]:
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

train['label'] = train['answer'].map(LABEL_MAP)
assert train['label'].notna().all(), 'an answer value is outside A to E'

print('train:', train.shape, '| test:', test.shape)
train.head(3)

train: (2000, 9) | test: (500, 7)


,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2


## 3. Exploratory data analysis

### 3.1 Label balance and text length

If one answer letter dominates, a model that always predicts it becomes the baseline that every
MAP@3 number has to be read against.

In [5]:
share = train['answer'].value_counts(normalize=True).sort_index()
print(share.round(4).to_string())
print('\nmost common letter:', share.idxmax(), f'at {share.max():.1%}')

answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620

most common letter: B at 24.5%


In [6]:
words=train['prompt'].str.split().str.len()+train[OPTIONS].apply(lambda r: sum(len(str(v).split()) for v in r), axis=1)
print(words.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())

count    2000.0
mean      149.5
std        85.4
min        15.0
50%       133.0
90%       257.1
99%       412.0
max       457.0


The median row is 133 words and the tail runs past 400. Subword tokenisation gives roughly
1.4 tokens per word, so `MAX_LEN = 512` is set to cover the tail. At 256 tokens the later options
would be cut off, and a truncated option can never be chosen correctly.

### 3.2 Duplicate rows

The row count is 2000, but that does not mean 2000 distinct questions. Checking for exact
duplicates across the prompt and all five options is the first test.

In [7]:
dupe_mask = train.duplicated(subset=['prompt'] + OPTIONS, keep=False)
print('rows that are part of an exact duplicate group:', int(dupe_mask.sum()))
print('unique prompts:', train['prompt'].nunique(), 'out of', len(train), 'rows')

for _, r in train[dupe_mask].sort_values('prompt').head(6).iterrows():
    print(repr(r['prompt'][:110]))

rows that are part of an exact duplicate group: 344
unique prompts: 1758 out of 2000 rows
'Choose the correct answer: What are permutation-inversion groups?'
'Choose the correct answer: What are permutation-inversion groups?'
'Choose the correct answer: What are permutation-inversion groups?'
'Choose the correct answer: What are permutation-inversion groups? based on the given context.'
'Choose the correct answer: What are permutation-inversion groups? based on the given context.'
'Choose the correct answer: What are the four qualitative levels of crystallinity described by geologists?'


### 3.3 Where the duplicates come from

Looking at a duplicated group shows the pattern. The same question and the same five options
appear more than once, and also sometimes can have different phrasing before the colon. Stripping the colon off to get exactly what the question is can give a genuine count of distinct questions.

In [8]:
train['core_q'] = train['prompt'].str.replace(r'^[^:]{0,60}:\s*','',regex=True).str.strip()
print('rows                 :', len(train))
print('unique prompts       :', train['prompt'].nunique())
print('unique core questions:', train['core_q'].nunique())

rows                 : 2000
unique prompts       : 1758
unique core questions: 1125


A smaller pool of questions was wrapped in several instruction phrases to reach 2000 rows. This
matters for the split: if one phrasing of a question lands in train and another lands in
validation, the model sees the answer during training and the validation score is meaningless.

### 3.4 Cleaning

Exact duplicates are dropped since they carry no extra information. Rows that share a core
question but differ in their options are kept, because the option sets genuinely differ. Those
rows are held together by the split instead.

In [9]:
before = len(train)
train = train.drop_duplicates(subset=['prompt'] + OPTIONS, keep='first').reset_index(drop=True)
print(f'dropped {before - len(train)} exact duplicates, {len(train)} rows remain')

dropped 183 exact duplicates, 1817 rows remain


## 4. Train and validation split

`GroupShuffleSplit` keeps every row sharing a core question on the same side. The assertion is
the point of the cell: it fails loudly if any question leaks across, rather than letting an
inflated score go unnoticed.

In [10]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(gss.split(train, groups=train['core_q']))

train_df = train.iloc[tr_idx].reset_index(drop=True)
val_df   = train.iloc[va_idx].reset_index(drop=True)

assert not set(train_df['core_q']) & set(val_df['core_q']), 'a core question leaked across the split'

print('train:', len(train_df), '| val:', len(val_df))
print('\nvalidation label balance')
print(val_df['answer'].value_counts(normalize=True).sort_index().round(3).to_string())

train: 1473 | val: 344

validation label balance
answer
A    0.212
B    0.262
C    0.215
D    0.134
E    0.177


The split is not exactly 80 to 20 because groups differ in size. Correct grouping matters more
than a round ratio.

`GroupShuffleSplit` does not stratify by label, so the balance is printed as a check. Every
letter sitting in a reasonable band means the validation set is usable as it stands.

## 5. The evaluation metric

MAP@3 gives full credit when the correct answer is ranked first, half when it is second, and a
third when it is third. Nothing below that scores. Every model in this notebook is measured with
the same function so the numbers are comparable.

In [11]:
def apk(actual, predicted, k=3):
    """Average precision at k for a single row with one correct answer."""
    predicted = predicted[:k]
    return 1 / (predicted.index(actual) + 1) if actual in predicted else 0.0

def map_at_3(predictions, labels):
    """Mean of apk over all rows."""
    return float(np.mean([apk(a, p) for a, p in zip(labels, predictions)]))

def top3(probs):
    """Rank the five option letters by probability and return the best three."""
    return [OPTIONS[i] for i in np.argsort(-np.asarray(probs))[:3]]

assert apk('B', ['B', 'A', 'C']) == 1.0
assert apk('B', ['A', 'B', 'C']) == 0.5
assert abs(apk('B', ['A', 'C', 'B']) - 1/3) < 1e-9
assert apk('B', ['A', 'C', 'D']) == 0.0
assert map_at_3([['B', 'A', 'C'], ['A', 'B', 'C']], ['B', 'B']) == 0.75
print('metric checks passed')

metric checks passed


## 6. Model 1: BiLSTM built from scratch

Every weight in this model starts random and is trained here. No pretrained weights are loaded.

The question and its five options are joined into one sequence and the task becomes five way
classification, so the loss is plain cross entropy. The architecture is an embedding layer, a two
layer bidirectional LSTM, masked mean pooling over the timesteps, and a small head that outputs
five scores.

This model is the reference point. Whatever it scores is what the task looks like with no outside
knowledge, and the pretrained models in section 8 are measured against it.

### 6.1 Tokenisation

The DeBERTa tokenizer turns text into ids. Only the tokenizer is reused. No DeBERTa weights are
loaded and the embedding table below is random at initialisation. Subword tokenisation handles
scientific vocabulary better than a word level vocabulary built from roughly 1400 training rows,
where most technical terms would appear once or not at all.

The question goes in as the first segment and the five options concatenated go in as the second,
which is the sentence pair form the tokenizer already supports.

In [12]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
assert tok.pad_token_id == 0, f'pad id is {tok.pad_token_id}, update the mask in the model'

def make_dataset(df, with_labels=True):
    """Tokenise a dataframe into a TensorDataset of ids, mask and optional labels."""
    questions = df['prompt'].astype(str).tolist()
    options   = [' '.join(f'{o}) {r[o]}' for o in OPTIONS) for _, r in df.iterrows()]

    enc = tok(questions, options, padding='max_length', truncation=True,
              max_length=MAX_LEN, return_tensors='pt')

    tensors = [enc['input_ids'], enc['attention_mask']]
    if with_labels:
        tensors.append(torch.tensor(df['label'].values, dtype=torch.long))
    return TensorDataset(*tensors)

print('vocabulary size:', len(tok))

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

vocabulary size: 128001


In [13]:
if BUILD:
    train_ds=make_dataset(train_df)
    val_ds=make_dataset(val_df)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    train_eval=DataLoader(train_ds, batch_size=BATCH_SIZE)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    print('batches, train:', len(train_loader), '| train_eval:', len(train_eval), '| val:', len(val_loader))

batches, train: 47 | train_eval: 47 | val: 11


### 6.2 Architecture

Real sequence lengths come from the attention mask and are passed to `pack_padded_sequence`, so
the LSTM never reads padding. Pooling is a masked mean over valid timesteps, which keeps padded
positions out of the average.

In [14]:
class BiLSTMClassifier(nn.Module):
    """Embedding, bidirectional LSTM, masked mean pooling, five way classifier head."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=5,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embed_dim, 
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes))

    def forward(self, input_ids, lengths):
        """Return one logit per option, shape batch by five."""
        embed = self.embed_dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(embed, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        rnn_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
        mask = (input_ids != 0).unsqueeze(-1)
        pooled = (rnn_out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled)

# shape check before any training
_m = BiLSTMClassifier(vocab_size=1000, embed_dim=32, hidden_dim=64)
_out = _m(torch.randint(1, 1000, (4, 20)), torch.tensor([20, 15, 10, 5]))
assert _out.shape == (4, 5), f'expected (4, 5), got {tuple(_out.shape)}'
print('forward pass shape check passed:', tuple(_out.shape))
del _m, _out

forward pass shape check passed: (4, 5)


### 6.3 Training

A plain PyTorch loop, one function to train an epoch and one to evaluate. Accuracy, macro F1 and
MAP@3 are computed every epoch and logged to Weights and Biases. The checkpoint is written only
when validation MAP@3 improves.

In [15]:
def train_one_epoch(model, loader, optimizer, device):
    """Run one training epoch and return the mean loss."""
    model.train()
    total = 0.0
    for input_ids, mask, labels in loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(input_ids, mask.sum(1)), labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)



@torch.no_grad()
def predict_probs(model, loader):
    """Return the probability array for every row in a loader, in loader order."""
    model.eval()
    batches = [F.softmax(model(b[0].to(device), b[1].sum(1)), dim=1).cpu().numpy()
               for b in loader]
    return np.concatenate(batches)


def score(model, loader, answers):
    """Accuracy, macro F1 and MAP@3 for a loader, given the true answer letters."""
    probs = predict_probs(model, loader)
    preds = [OPTIONS[i] for i in probs.argmax(1)]
    return (accuracy_score(answers, preds),
            f1_score(answers, preds, average='macro'),
            map_at_3([top3(p) for p in probs], answers))

In [16]:
if BUILD:
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret('WANDB'))
    except Exception as e:
        print('W&B login skipped:', e)

    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='bilstm-scratch-grouped-viva2', group='final-notebook',
                     config={'model': 'biLSTM from scratch', 'embed_dim': EMBED_DIM,
                             'hidden_dim': HIDDEN_DIM, 'max_len': MAX_LEN, 'lr': LR,
                             'split': 'GroupShuffleSplit on core_q'}, reinit=True)
    
    bilstm = BiLSTMClassifier(len(tok), EMBED_DIM, HIDDEN_DIM,
                              pad_idx=tok.pad_token_id).to(device)
    print('trainable parameters:', f'{sum(p.numel() for p in bilstm.parameters()):,}')

    optimizer = torch.optim.AdamW(bilstm.parameters(), lr=LR)
    tr_ans = train_df['answer'].tolist()
    va_ans = val_df['answer'].tolist()
    best_map3 = -1.0

    for epoch in range(N_EPOCHS):
        loss = train_one_epoch(bilstm, train_loader, optimizer, device)
        tr_acc, _, tr_map3 = score(bilstm, train_eval, tr_ans)
        va_acc, va_f1, va_map3 = score(bilstm, val_loader, va_ans)

        run.log({'epoch': epoch, 'train_loss': loss, 'train_acc': tr_acc,
                 'train_map3': tr_map3, 'val_acc': va_acc,
                 'val_f1_macro': va_f1, 'val_map3': va_map3})
        print(f'epoch {epoch:2d}  loss {loss:.4f}  train_acc {tr_acc:.4f}  '
              f'val_acc {va_acc:.4f}  val_f1 {va_f1:.4f}  val_map3 {va_map3:.4f}')

        if va_map3 > best_map3:
            best_map3 = va_map3
            torch.save(bilstm.state_dict(), f'{WORK}/bilstm_best.pt')

    run.log({'best_val_map3': best_map3})
    run.finish()
    print('best validation MAP@3:', round(best_map3, 4))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


trainable parameters: 18,884,229
epoch  0  loss 1.4162  train_acc 0.4902  val_acc 0.3895  val_f1 0.3186  val_map3 0.5538
epoch  1  loss 0.6834  train_acc 0.9457  val_acc 0.8605  val_f1 0.8574  val_map3 0.9157
epoch  2  loss 0.2310  train_acc 0.9912  val_acc 0.9593  val_f1 0.9593  val_map3 0.9695
epoch  3  loss 0.0612  train_acc 0.9980  val_acc 0.9884  val_f1 0.9882  val_map3 0.9937
epoch  4  loss 0.0411  train_acc 0.9946  val_acc 0.9506  val_f1 0.9521  val_map3 0.9753
epoch  5  loss 0.0556  train_acc 0.9939  val_acc 0.9797  val_f1 0.9791  val_map3 0.9898
epoch  6  loss 0.0287  train_acc 1.0000  val_acc 0.9971  val_f1 0.9975  val_map3 0.9985
epoch  7  loss 0.0090  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch  8  loss 0.0014  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch  9  loss 0.0060  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000
epoch 10  loss 0.0213  train_acc 1.0000  val_acc 1.0000  val_f1 1.0000  val_map3 1.0000

best_val_map3,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▇██████████████████
train_loss,█▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_map3,▁▇██████████████████
val_acc,▁▆██▇███████▇███████
val_f1_macro,▁▇██████████▇███████
val_map3,▁▇██████████████████
best_val_map3,1
epoch,19
train_acc,1


best validation MAP@3: 1.0


Train accuracy is logged next to validation accuracy on purpose. A high training score with a
lower validation score is ordinary overfitting on a small dataset. The two rising together to a
perfect score is a different thing and means information is reaching validation.

An earlier version of this model did reach a perfect validation score. The cause was the split,
not the architecture. Splitting on raw prompts left the same question, wrapped in a different
instruction phrase, on both sides. Grouping on `core_q` in section 4 is what fixes it, and it is
why the assertion in that cell exists.

### 6.4 Test predictions from the from scratch model

In [17]:
if BUILD:
    bilstm.load_state_dict(torch.load(f'{WORK}/bilstm_best.pt'))

    test_loader = DataLoader(make_dataset(test, with_labels=False), batch_size=BATCH_SIZE)
    test_probs_bilstm = predict_probs(bilstm, test_loader)

    np.save(f'{WORK}/test_probs_bilstm.npy', test_probs_bilstm)
    print('saved test_probs_bilstm.npy', test_probs_bilstm.shape)

    del bilstm, train_ds, val_ds, train_loader, train_eval, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

saved test_probs_bilstm.npy (500, 5)


## 7. The Wikipedia corpus and retrieval

These are science questions whose answers are facts, not things that can be worked out from the
wording. The similarity baselines in the earlier milestones lost to the majority class floor
because the distractors are written to share vocabulary with the question, so word overlap does
not separate them. Supplying the model with the passage that contains the fact is what changes
that.

The corpus is a July 2023 Wikipedia dump with a prebuilt FAISS index over MiniLM embeddings. The
index holds one vector per article and the article text sits in parquet shards alongside it.
Retrieval runs in three stages: FAISS finds candidate articles, the articles are split into
chunks, and a cross encoder picks the five chunks that become the context.

In [18]:
if BUILD:
    import faiss
    from sentence_transformers import SentenceTransformer, CrossEncoder

    print('loading the FAISS index, about a minute')
    wiki_index = faiss.read_index(WIKI_INDEX)
    wiki_map   = pd.read_parquet(WIKI_MAP)

    MAP_ID, MAP_FILE = 'id', 'file'
    ART_COLS = ['id', 'title', 'text']
    assert {MAP_ID, MAP_FILE} <= set(wiki_map.columns), f'map columns are {list(wiki_map.columns)}'

    bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=device)
    reranker   = CrossEncoder(RERANKER_NAME, device=device)

    NEED_RETRIEVAL = not os.path.exists(CTX_CACHE)

    print('article vectors :', f'{wiki_index.ntotal:,}')
    print('dimension       :', wiki_index.d)
    print('parquet shards  :', len([f for f in os.listdir(WIKI_DIR) if f.endswith('.parquet')]))
    print('retrieval needed:', NEED_RETRIEVAL)

loading the FAISS index, about a minute


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

article vectors : 6,286,775
dimension       : 384
parquet shards  : 29
retrieval needed: True


### 7.1 Stage 1, find candidate articles

Each option is joined to the question to form its own query, so an option naming a specific term
can pull in the article about that term even when the question never mentions it. Five queries
per question, ten articles each, giving fifty candidates before deduplication.

Retrieval is run over the validation rows and the test rows together. Positions zero to `N_VAL`
are validation rows in split order and the rest are test rows in file order.

In [19]:
if BUILD:
    all_rows = [val_df.iloc[i] for i in range(len(val_df))] + \
               [test.iloc[i]   for i in range(len(test))]
    N_VAL = len(val_df)

if NEED_RETRIEVAL:
    queries = [f'{r["prompt"]} {r[o]}' for r in all_rows for o in OPTIONS]

    q_emb = bi_encoder.encode(queries, batch_size=64, show_progress_bar=True)
    _, art = wiki_index.search(np.asarray(q_emb, dtype='float32'), K_ARTICLES)
    article_rows = art.reshape(len(all_rows), 5 * K_ARTICLES)

    print('questions:', len(all_rows), '| candidates each:', article_rows.shape[1])
    
    needed   = wiki_map.iloc[article_rows.flatten()]
    by_shard = needed.groupby(MAP_FILE)[MAP_ID].apply(lambda s: set(s.astype(str))).to_dict()
    wanted   = set(needed[MAP_ID].astype(str))
    map_ids  = wiki_map[MAP_ID].astype(str).to_numpy()

    del wiki_index, wiki_map, q_emb, art, needed
    gc.collect()
    print('freed the FAISS index and the article map')

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

questions: 844 | candidates each: 50
freed the FAISS index and the article map


### 7.2 Stage 2, load and chunk the article text

Only the retrieved articles are read, and only three columns of them. The full dump does not fit
in memory, so the shards are read one at a time and filtered down to the ids that were actually
retrieved before anything is kept.

Articles are split with `RecursiveCharacterTextSplitter`, which prefers to break at paragraph and
sentence boundaries rather than at a fixed character count, and overlaps consecutive chunks so a
fact spanning a boundary is not lost. Each chunk is prefixed with the article title, since a
chunk taken from the middle of an article often never repeats the subject by name.

In [20]:
if NEED_RETRIEVAL:
    import pyarrow.parquet as pq
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800, chunk_overlap=150, separators=['\n\n', '\n', '. ', ' '])

    MAX_ARTICLE_CHARS, MAX_CHUNKS = 30000, 40

    article_chunks = {}
    for shard, ids in tqdm(by_shard.items(), desc='parquet shards'):
        pf = pq.ParquetFile(os.path.join(WIKI_DIR, str(shard)))
        for batch in pf.iter_batches(batch_size=2000, columns=ART_COLS):
            bdf = batch.to_pandas()
            hit = bdf[bdf['id'].astype(str).isin(ids)]
            for _, r in hit.iterrows():
                text = str(r['text'])[:MAX_ARTICLE_CHARS]
                article_chunks[str(r['id'])] = [f'{r["title"]}. {c}'
                                                for c in splitter.split_text(text)][:MAX_CHUNKS]
            del bdf, hit
        del pf
        gc.collect()

    cov = len(wanted & set(article_chunks)) / len(wanted)
    print(f'article coverage: {cov:.1%}')
    assert cov > 0.95, 'coverage below 95 percent, ids are not matching between index and parquet'

parquet shards:   0%|          | 0/28 [00:00<?, ?it/s]

article coverage: 100.0%


### 7.3 Stage 3, pick the best chunks

Article level retrieval is too coarse to use directly, since most of a long article is irrelevant
to any one question. All chunks from the candidate articles are pooled and narrowed in two steps.

The bi encoder scores the question against every chunk independently, which is fast because the
chunk embeddings do not depend on the query, and keeps the best twenty. The cross encoder then
reads each question and chunk together in a single pass and reorders those twenty, which is far
more accurate and far slower, so it is only worth running on a shortlist.

Milestone 3 measured a 73 percent top five hit rate for the bi encoder on its own. The reranker
is the stage that lifts that, at the cost of one extra forward pass per shortlisted chunk.

In [22]:
if NEED_RETRIEVAL:
    from sentence_transformers import util

    contexts = []
    for qi, row in enumerate(tqdm(all_rows, desc='retrieve')):
        ids = pd.unique(article_rows[qi])
        chunks = [c for i in ids for c in article_chunks.get(map_ids[i], [])]
        if not chunks:
            contexts.append('')
            continue

        q_text = str(row['prompt']) + ' ' + ' '.join(str(row[o]) for o in OPTIONS)

        sims = util.cos_sim(bi_encoder.encode(q_text), bi_encoder.encode(chunks, batch_size=128))[0]
        shortlist = [chunks[i] for i in sims.topk(min(K_RERANK, len(chunks))).indices]

        ranked = reranker.rank(q_text, shortlist, top_k=K_FINAL)
        contexts.append(' '.join(shortlist[r['corpus_id']] for r in ranked)[:CTX_CHARS])

    with open(CTX_CACHE, 'wb') as f:
        pickle.dump(contexts, f)

retrieve:   0%|          | 0/844 [00:00<?, ?it/s]

In [23]:
if BUILD:
    contexts = pickle.load(open(CTX_CACHE, 'rb'))
    assert len(contexts) == N_VAL + len(test), 'context count does not match the row count'

    print('contexts:', len(contexts), '| empty:', sum(1 for c in contexts if not c))
    print('\nvalidation row 0 context')
    print(contexts[0][:400], '...')

contexts: 844 | empty: 0

validation row 0 context
Heavy ion fusion. Heavy ion fusion is a fusion energy concept that uses a stream of high-energy ions from a particle accelerator to rapidly heat and compress a small pellet of fusion fuel. It is a subclass of the larger inertial confinement fusion (ICF) approach, replacing the more typical laser systems with an accelerator. Accelerators have the potential to be much more efficient in terms of deli ...


`contexts_final.pkl` is tied to the current split. Positions zero to `N_VAL` are validation rows
in their split order, so changing the split or the seed means deleting this file and regenerating
it. Left in place, the contexts would line up with the wrong questions and every retrieval number
below would be wrong in a way that is hard to see.

In [25]:
if BUILD:
    for name in ['bi_encoder', 'reranker', 'article_chunks', 'article_rows', 'map_ids']:
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()

    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f'GPU {d}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')

GPU 0: 15.0 GB free of 15.6 GB
GPU 1: 15.5 GB free of 15.6 GB


## 8. Model 2 and 3: Llama 3.1 8B, retrieval and the ensemble

The base model is Llama 3.1 8B Instruct, loaded in 4 bit NF4 with float16 compute and sharded
across both T4s. Float16 is used rather than bfloat16 because the T4 has no native bfloat16
support. Quantisation is what makes an 8B model fit at all here.

The Milestone 4 QLoRA adapter is pulled through kagglehub. When it loads, four signals are
available. When it does not, the notebook continues with the two base signals rather than
stopping.

In [26]:
if BUILD:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    bnb = BitsAndBytesConfig(load_in_4bit=True,
                             bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)

    llama_tok = AutoTokenizer.from_pretrained(LLAMA_NAME)
    llama_tok.pad_token = llama_tok.pad_token or llama_tok.eos_token

    llama = AutoModelForCausalLM.from_pretrained(
        LLAMA_NAME, quantization_config=bnb, device_map='auto', low_cpu_mem_usage=True)
    llama.eval()
    print('loaded', LLAMA_NAME, '| sharded over:', sorted(set(llama.hf_device_map.values())))

    if USE_ADAPTER:
        import kagglehub
        from peft import PeftModel
        llama = PeftModel.from_pretrained(llama, kagglehub.model_download(ADAPTER_HANDLE))
        llama.eval()
        print('QLoRA adapter attached')

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

loaded meta-llama/Llama-3.1-8B-Instruct | sharded over: [0, 1]
QLoRA adapter attached


### 8.1 Reading answers from logits

The model is never asked to generate text. The prompt ends at `Answer:` and the logits for the
next token are read directly. Only the five entries corresponding to A to E matter, so those are
taken and softmaxed into a distribution over the options. One forward pass per signal, no
sampling, and the full ranking needed for MAP@3 comes out of it.

Each letter has two token ids, one bare and one with a leading space. The larger logit of the two
is used.

In [27]:
if BUILD:
    letter_ids = {o: list({llama_tok.encode(o, add_special_tokens=False)[-1],
                           llama_tok.encode(' ' + o, add_special_tokens=False)[-1]})
                  for o in OPTIONS}
    assert all(letter_ids[o] for o in OPTIONS), 'a letter produced no token id'
    print('letter token ids:', letter_ids)

    @torch.no_grad()
    def letter_probs(row, context, use_adapter):
        """Probability of each option letter for one question, from the next token logits."""
        opts = '\n'.join(f'{o}. {row[o]}' for o in OPTIONS)
        head = f'Context:\n{context}\n\n' if context else ''
        prompt = f'{head}Question: {row["prompt"]}\n\nOptions:\n{opts}\n\nAnswer:'

        ids = llama_tok(prompt, return_tensors='pt').input_ids.to(llama.device)
        if USE_ADAPTER and not use_adapter:
            with llama.disable_adapter():
                logits = llama(ids).logits[0, -1]
        else:
            logits = llama(ids).logits[0, -1]

        raw = torch.tensor([max(logits[i].item() for i in letter_ids[o]) for o in OPTIONS])
        return torch.softmax(raw, dim=0).numpy()

    demo = letter_probs(all_rows[0], contexts[0], False)
    print('validation row 0 ->', top3(demo), '| true answer', all_rows[0]['answer'])

letter token ids: {'A': [32, 362], 'B': [33, 426], 'C': [34, 356], 'D': [35, 423], 'E': [36, 469]}
validation row 0 -> ['A', 'D', 'C'] | true answer A


### 8.2 The four signals

Two variables give four combinations: with or without retrieved context, and base or fine tuned
weights.

| Signal | Context | Weights |
|---|---|---|
| base_rag | retrieved | base |
| base_closed | none | base |
| tuned_rag | retrieved | QLoRA adapter |
| tuned_closed | none | QLoRA adapter |

Ensembling works when the members make different mistakes. Here the differences are real: the
closed book signals rely on what the model already knows, the RAG signals rely on the retrieved
passage, and the tuned signals have seen the answer format during fine tuning. Averaging the four
probability arrays lets a confident signal outvote an uncertain one.

In [28]:
if BUILD:
    SIGNALS = ['base_closed', 'base_rag'] + (['tuned_closed', 'tuned_rag'] if USE_ADAPTER else [])
    CONTAMINATED = {'tuned_closed', 'tuned_rag'}
    print(f'{len(SIGNALS)} signals x {len(all_rows)} rows = '
          f'{len(SIGNALS) * len(all_rows):,} forward passes')

    sig = {s: [] for s in SIGNALS}
    for i, row in enumerate(tqdm(all_rows, desc='llama signals')):
        for s in SIGNALS:
            sig[s].append(letter_probs(row,
                                       contexts[i] if s.endswith('rag') else '',
                                       s.startswith('tuned')))

    sig = {s: np.array(v) for s, v in sig.items()}
    ens = np.mean([sig[s] for s in SIGNALS], axis=0)
    assert ens.shape == (len(all_rows), 5)

4 signals x 844 rows = 3,376 forward passes


llama signals:   0%|          | 0/844 [00:00<?, ?it/s]

In [29]:
if BUILD:
    val_true = [r['answer'] for r in all_rows[:N_VAL]]

    val_scores = {s: map_at_3([top3(p) for p in sig[s][:N_VAL]], val_true) for s in SIGNALS}
    val_scores['ensemble'] = map_at_3([top3(p) for p in ens[:N_VAL]], val_true)

    for name, v in val_scores.items():
        note = '   adapter saw these rows, not a valid estimate' if name in CONTAMINATED else ''
        print(f'{name:14s} val MAP@3 {v:.4f}{note}')

    for s in SIGNALS:
        np.save(f'{WORK}/test_probs_{s}.npy', sig[s][N_VAL:])
    np.save(f'{WORK}/test_probs_ensemble.npy', ens[N_VAL:])
    np.save(f'{WORK}/val_probs_ensemble.npy',  ens[:N_VAL])
    print('saved', len(SIGNALS) + 2, 'probability arrays')

base_closed    val MAP@3 0.8188
base_rag       val MAP@3 0.8832
tuned_closed   val MAP@3 0.9869   adapter saw these rows, not a valid estimate
tuned_rag      val MAP@3 0.9952   adapter saw these rows, not a valid estimate
ensemble       val MAP@3 0.9942
saved 6 probability arrays


### 8.3 Log to Weights and Biases

In [31]:
if BUILD:
    import wandb
    run = wandb.init(entity='22f3000114-dl-genai-project', project='dl-genai-project',
                     name='llama-ensemble-final', group='final-notebook',
                     config={'base': LLAMA_NAME, 'quantisation': '4bit nf4, fp16 compute',
                             'adapter': USE_ADAPTER,
                             'retrieval': 'MiniLM bi encoder, ms-marco cross encoder rerank',
                             'k_articles': K_ARTICLES, 'k_rerank': K_RERANK,
                             'k_final': K_FINAL,
                             'split': 'GroupShuffleSplit on core_q'}, reinit=True)
    run.log({f'val_map3_{k}': v for k, v in val_scores.items()})
    run.finish()
    print('logged to W&B')

val_map3_base_closed,▁
val_map3_base_rag,▁
val_map3_ensemble,▁
val_map3_tuned_closed,▁
val_map3_tuned_rag,▁
val_map3_base_closed,0.8188
val_map3_base_rag,0.88324
val_map3_ensemble,0.99419
val_map3_tuned_closed,0.98692
val_map3_tuned_rag,0.99516


logged to W&B


## 9. Model comparison

All three models are scored on the same validation split with the same MAP@3 function, which is
what makes the numbers comparable at all.

Two reference points sit in the table alongside them. The constant baseline always predicts the
three most common answer letters and represents what you get from the label distribution with no
model at all. The BiLSTM represents what the task looks like with a model but no pretrained
knowledge. Anything a pretrained model adds is the distance above those two.

The `trusted` column marks the rows whose validation score is a valid estimate. The tuned signals
and therefore the ensemble containing them are excluded, because the M4 adapter was trained
before the duplicate question fix in section 3 and has seen these validation rows.

In [34]:
if BUILD:
    val_share = val_df['answer'].value_counts(normalize=True)
    top_three = val_share.sort_values(ascending=False).index[:3]
    baseline_map3 = sum(val_share[l] * w for l, w in zip(top_three, [1, 0.5, 1/3]))
    print('constant baseline predicts', ' '.join(top_three), f'-> MAP@3 {baseline_map3:.4f}')

    rows = [{'model': 'Constant baseline', 'type': 'baseline',
             'val_map3': baseline_map3},
            {'model': 'BiLSTM from scratch', 'type': 'scratch',
             'val_map3': best_map3}]

    rows += [{'model': f'Llama 3.1 8B, {s}', 'type': 'pretrained',
              'val_map3': val_scores[s]}
             for s in SIGNALS]

    rows.append({'model': f'{len(SIGNALS)} signal ensemble', 'type': 'ensemble',
                 'val_map3': val_scores['ensemble']})

    pd.DataFrame(rows).sort_values('val_map3', ascending=False) \
                      .reset_index(drop=True) \
                      .to_csv(f'{WORK}/model_comparison.csv', index=False)

constant baseline predicts B C A -> MAP@3 0.4399


In [37]:
comparison = pd.read_csv(f'{os.path.join(ART, "model_comparison.csv")}')

print('all rows')
print(comparison.round(4).to_string(index=False))

all rows
                     model       type  val_map3
       BiLSTM from scratch    scratch    1.0000
   Llama 3.1 8B, tuned_rag pretrained    0.9952
         4 signal ensemble   ensemble    0.9942
Llama 3.1 8B, tuned_closed pretrained    0.9869
    Llama 3.1 8B, base_rag pretrained    0.8832
 Llama 3.1 8B, base_closed pretrained    0.8188
         Constant baseline   baseline    0.4399


## 10. Submission

Submissions read the saved arrays. Set `SUBMIT_MODEL` at the top, run with `BUILD = False`, and
this section writes `submission.csv`. Because the arrays are fixed on disk, the same selection
always produces the same file.

Three submissions are planned: `bilstm, base_rag, ensemble, base_closed, tuned_closed, tuned_rag`

In [38]:
SUBMIT_FILES = {
    'bilstm':   'test_probs_bilstm.npy',
    'base_rag': 'test_probs_base_rag.npy',
    'ensemble': 'test_probs_ensemble.npy',
    'base_closed': 'test_probs_base_closed.npy',
    'tuned_closed': 'test_probs_tuned_closed.npy',
    'tuned_rag': 'test_probs_tuned_rag.npy'
}

path = os.path.join(ART, SUBMIT_FILES[SUBMIT_MODEL])
assert os.path.exists(path), f'{path} not found, run with BUILD = True or point ART at the artifacts'

probs = np.load(path)
assert probs.shape == (len(test), 5), f'expected {(len(test), 5)}, found {probs.shape}'

submission = pd.DataFrame({'id': test['id'],
                           'prediction': [' '.join(top3(p)) for p in probs]})

assert submission['prediction'].str.split().map(lambda p: len(set(p)) == 3).all(), \
    'a prediction does not contain three distinct letters'

submission.to_csv('submission.csv', index=False)
print('model :', SUBMIT_MODEL)
print('source:', path)
print('rows  :', len(submission))
submission.head()

model : base_rag
source: /kaggle/working/test_probs_base_rag.npy
rows  : 500


,id,prediction
0,1,A D E
1,2,B A D
2,3,B D C
3,4,A E C
4,5,C A B


## 11. Notes

### What is in this notebook

Three models of genuinely different architectures. The BiLSTM is written and trained here with no pretrained weights. Llama 3.1 8B is a pretrained decoder used through logit extraction rather than generation. The ensemble averages four signals that differ in both context and weights.

### The data problem and how it was handled

The training file has 2000 rows but around 1125 distinct questions, because a smaller pool was wrapped in several instruction phrases. Splitting rows independently puts the same question on both sides and inflates every validation score. Section 4 groups on the stripped question text and asserts zero overlap. This was found by noticing a perfect validation score that no honest model should reach on this task.

### Known limitations

The M4 QLoRA adapter predates the split fix, so the two tuned signals and the ensemble containing them have seen the validation rows. Their scores are upper bounds and section 9 marks them.

Retrieved articles are often correct in topic but the stored text is a truncated intro paragraph, which frequently lacks the detail separating the distractors. 

Contexts are capped at 4500 characters. The validation set is small enough that differences of one or two points are within noise. 

The RAG dataset is from 2023, does not account for changes up to date.

### Milestone mapping

Milestone 1 is the EDA and baselines in section 3. Milestone 2 is the similarity baselines that motivated retrieval. Milestone 3 is the retrieval pipeline in section 7. Milestone 4 is the QLoRA adapter behind the tuned signals. Milestone 5 is the ensembling and MAP@3 work in section 8.